# 14 - Contextual chunk prefixes: the document side of the vocabulary gap

> **Run order.** This notebook is step 14 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` - the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.

[ADR-007](../docs/adr/0007-retrieval-strategy.md) closed with a prediction:

> The answer element carries no company name, no fiscal year, and no statement
> title - a chunk prefix of `SUNPHARMA FY2024 - Statement of Profit and Loss`
> would give both halves of the retriever something to match.

Query expansion fixed the **question** side of the vocabulary gap. This notebook
tests the **document** side. Two thirds of that prediction survived measurement;
one third did not, and finding out which is the point of the first section.

## 1. What the chunks actually carry

Before building anything: `heading` is already prepended to every chunk, so the
prefix only earns its place for context that is genuinely missing.

In [ ]:
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

from analyst import evaluation as ev
from analyst.config import get_settings
from analyst.indexing import load_chunks

settings = get_settings()
questions = ev.load_questions(settings.data_dir / "benchmark" / "questions.jsonl")
print(f"{len(questions)} questions   benchmark {ev.bench_sha(questions)}")

base = load_chunks(with_context=False)
print(f"{len(base):,} chunks")

# Which of ADR-007's three claims hold?
have_heading = sum(1 for c in base if c.heading)
have_ticker = sum(1 for c in base if c.ticker.lower() in c.text.lower())
have_fy = sum(1 for c in base if str(c.fiscal_year) in c.text)
pd.DataFrame([
    {"context": "a heading", "chunks": have_heading, "share": have_heading / len(base)},
    {"context": "its own company", "chunks": have_ticker, "share": have_ticker / len(base)},
    {"context": "its fiscal year", "chunks": have_fy, "share": have_fy / len(base)},
]).assign(share=lambda d: (d["share"] * 100).round(1))

**The company is missing, the year mostly is not, and a heading is nearly always
present.** So the interesting question is what those headings *say*.

In [ ]:
from collections import Counter

heads = Counter(c.heading for c in base if c.heading)
reuse = pd.Series(list(heads.values()))
print(f"distinct headings: {len(heads):,}   reuse p50={reuse.median():.0f} "
      f"p90={reuse.quantile(0.9):.0f} max={reuse.max()}")
print()
for h, n in heads.most_common(8):
    flat = " / ".join(h.splitlines())
    print(f"{n:>5}x  {flat[:76]}")

### These are running page bands, not section titles

`226 / Statutory Reports / Corporate Overview / Financial Statements` is printed
on every page of a section. One heading repeats on **507 chunks**. A string
identical across hundreds of chunks cannot help tell them apart - it only
dilutes the vector and spends the encoder's 512-token budget.

`analyst.chunking.clean_heading` drops them, matching **whole lines only**, so
`CONSOLIDATED FINANCIAL STATEMENTS OF ICICI BANK LIMITED` - which names the
company - survives while a bare `Financial Statements` band does not.

In [ ]:
from analyst.chunking import clean_heading

for h in ["226\nStatutory Reports\nCorporate Overview\nFinancial Statements",
          "CONSOLIDATED FINANCIAL STATEMENTS OF ICICI BANK LIMITED",
          "212\nConsolidated Balance Sheet\nStatutory Reports",
          "Integrated Annual Report 2024-25\n53"]:
    print(f"{' / '.join(h.splitlines())[:58]:<60} ->  {clean_heading(h)}")

## 2. The part of ADR-007's prediction that did not survive

The prediction wanted a **statement title** in the prefix. Statement titles do
exist in the elements, so the obvious move is to scan backwards for the nearest
one. That was measured before it was built - and it does not work.

In [ ]:
import re

from sqlalchemy import select

from analyst.db import session_scope
from analyst.models import Document, ElementRow

TITLE = re.compile(
    r"(statement of profit and loss|balance sheet|cash flow statement"
    r"|statement of cash flows|profit and loss account|statement of changes in equity)",
    re.IGNORECASE,
)

with session_scope() as s:
    order = {}
    for d in s.execute(select(Document)).scalars().all():
        rows = s.execute(
            select(ElementRow.element_id, ElementRow.page, ElementRow.text)
            .where(ElementRow.document_id == d.document_id)
            .order_by(ElementRow.page, ElementRow.seq)).all()
        order[d.document_id] = [(r[0], r[1], r[2] or "") for r in rows]

pos = {eid: (doc, i) for doc, els in order.items() for i, (eid, _, _) in enumerate(els)}

rows = []
for q in questions:
    for eid in q.expected_element_ids:
        if eid not in pos:
            continue
        doc, i = pos[eid]
        els = order[doc]
        back, title = None, ""
        for j in range(i, max(-1, i - 60), -1):
            if TITLE.search(els[j][2][:150]):
                back, title = i - j, " ".join(els[j][2][:80].split())
                break
        rows.append({"back": back, "same_page": back is not None and els[i - back][1] == els[i][1],
                     "title": title})

t = pd.DataFrame(rows)
found = t["back"].notna()
print(f"answer elements: {len(t)}")
print(f"a statement title within 60 elements : {found.sum()} ({found.mean():.1%})")
print(f"   ... on the element's own page     : {t.loc[found, 'same_page'].mean():.1%}")
print(f"   ... median distance back          : {t.loc[found, 'back'].median():.0f} elements")
print("\nwhat the scan actually recovers:")
for x, n in Counter(t.loc[found, "title"]).most_common(6):
    print(f"   {n:>3}x  {x[:74]}")

**Rejected.** A title is found for only ~65% of answer elements, almost never on
the element's own page, a median of 21 elements back - and most matches are
*prose mentions*, not titles: "Refer consolidated statement of changes in equity
for detailed movement...". Attaching those would label chunks with confident,
wrong context more often than right context.

So the prefix carries **only what is derivable with certainty**: the company and
the year, in both the vocabulary the question uses and the vocabulary the filing
prints. `analyst.chunking.DocContext`.

In [ ]:
from analyst.chunking import DocContext

print(DocContext(ticker="SUNPHARMA", company="Sun Pharmaceutical Industries",
                 fiscal_year=2024).prefix)

## 3. A bug found on the way: chunks still over the encoder budget

A prefix is paid for out of the same 512 tokens as the passage, so the budget had
to be checked before adding to it. It was not being kept.

Day 3 capped the **row-packing** path at `MAX_TABLE_CHARS`. Three other paths had
no cap at all: a table whose JSON has no rows, a single text element larger than
the whole target, and - the expensive one - a table whose **header** is itself
oversized, since the header is repeated on every slice.

In [ ]:
from analyst.chunking import budget

over = [c for c in base if len(c.embed_text) > budget(c.type)]
print(f"chunks over the encoder budget: {len(over)}")
print("  (this notebook builds `base` AFTER the fix, so it reads 0 here)")

# The worst offender in the corpus, before the fix: a 3,328-character "header".
with session_scope() as s:
    tj = s.execute(select(ElementRow.table_json).where(
        ElementRow.element_id == "HDFCBANK-annual_report-FY2025-2a1879ee:p0052:e0000")).scalar()
from analyst.chunking import _table_parts
hdr, rws = _table_parts(tj)
print(f"\nworst table: {len(hdr)} header cells totalling "
      f"{len(' | '.join(hdr)):,} chars, over {len(rws)} rows")
print("A header that leaves no room for a row is not a header - it is content,")
print("and repeating it on every slice put 138 chunks past the encoder limit.")

## 4. Build both arms

Three things changed at once, so the experiment separates them:

| arm | budget fix | furniture stripped | context prefix |
|---|---|---|---|
| `fix` | yes | no | no |
| `ctx` | yes | yes | yes |

The old `elements_bge-small` collection is neither - it predates the budget fix.
Comparing `ctx` straight against it would credit the prefix with un-truncating
136 chunks, so `fix` exists to hold that constant.

In [ ]:
ctx_chunks = load_chunks(with_context=True)

rows = []
for label, cs in (("fix", base), ("ctx", ctx_chunks)):
    rows.append({
        "arm": label, "chunks": len(cs),
        "with heading": sum(1 for c in cs if c.heading),
        "over budget": sum(1 for c in cs if len(c.embed_text) > budget(c.type)),
        "median chars": int(pd.Series([len(c.embed_text) for c in cs]).median()),
        "max chars": max(len(c.embed_text) for c in cs),
    })
pd.DataFrame(rows)

In [ ]:
# What the encoder now reads for a chunk that answers a benchmark question.
sun = next(q for q in questions if q.ticker == "SUNPHARMA" and q.concept == "Total Revenue")
want = set(sun.expected_element_ids)
hit = next(c for c in ctx_chunks if want & set(c.element_ids))

print("QUESTION :", sun.question)
print("\nEMBEDDED :")
print(hit.embed_text[:300].replace("\n", " | "))
print("\nSTORED (what a citation quotes):")
print(hit.text[:160].replace("\n", " | "))

`embed_text` and `text` are deliberately different. The prefix is something we
assembled; quoting it back as though the filing said it would be a provenance
bug, so `text` stays verbatim and only the embedding sees the prefix.

## 5. Index

Four collections: both arms, dense and hybrid. ~30 minutes each - **run this
unattended.** Resumable: a complete collection is skipped, so a failure part way
through does not cost the whole sweep.

In [ ]:
import time

from analyst.retrievers import open_hybrid, open_store

MODEL = "bge-small"
BATCH = 256
FORCE = False

rows = []
for variant, cs in (("fix", base), ("ctx", ctx_chunks)):
    # --- dense
    embedder, store = open_store(settings, MODEL, variant)
    if FORCE or not store.exists() or store.count() != len(cs):
        store.recreate()
        t0 = time.perf_counter()
        for i in range(0, len(cs), BATCH):
            w = cs[i : i + BATCH]
            store.upsert(w, list(embedder.embed_documents([c.embed_text for c in w])))
        rows.append({"collection": store.collection, "points": store.count(),
                     "minutes": round((time.perf_counter() - t0) / 60, 1)})
    print(f"{store.collection:<28} {store.count():>6,} points")

    # --- hybrid (its own collection: Qdrant fixes vector layout at creation)
    embedder, sparse, hstore = open_hybrid(settings, MODEL, variant)
    if FORCE or not hstore.exists() or hstore.count() != len(cs):
        hstore.recreate()
        t0 = time.perf_counter()
        for i in range(0, len(cs), BATCH):
            w = cs[i : i + BATCH]
            texts = [c.embed_text for c in w]
            hstore.upsert(w, list(embedder.embed_documents(texts)),
                          list(sparse.embed_documents(texts)))
        rows.append({"collection": hstore.collection, "points": hstore.count(),
                     "minutes": round((time.perf_counter() - t0) / 60, 1)})
    print(f"{hstore.collection:<28} {hstore.count():>6,} points")

pd.DataFrame(rows) if rows else "all four collections already complete"

## 6. Measure

Same 44 questions, same injected-retriever scoring, same ledger. Query expansion
is on everywhere - ADR-007 adopted it, so it is the baseline this builds on, not
a variable.

In [ ]:
from analyst.retrievers import dense, hybrid


def record(label: str, search, points: int, notes: str) -> ev.Run:
    cfg = ev.RunConfig(retriever=label, model=MODEL, filters="ticker+year",
                       limit=max(ev.K_VALUES), points=points, notes=notes)
    run = ev.build_run(
        cfg,
        ev.evaluate(questions, search, limit=max(ev.K_VALUES)),
        questions,
        deep=ev.evaluate(questions, search, limit=max(ev.DEPTHS)),
        root=ev.ROOT,
    )
    ev.append_run(run)
    print(f"{label:<24} R@5 {run.metrics.recall_at[5]:.3f}   "
          f"ceiling@200 {run.depth_curve.get(200, 0):.3f}")
    return run


results = {}
for variant in ("fix", "ctx"):
    embedder, store = open_store(settings, MODEL, variant)
    _, sparse, hstore = open_hybrid(settings, MODEL, variant)
    note = "ADR-008 " + ("context prefix + furniture stripped" if variant == "ctx"
                         else "encoder-budget fix only")
    results[f"dense+expand[{variant}]"] = record(
        f"dense+expand[{variant}]",
        dense(embedder, store, "ticker+year", expand=True), store.count(), note)
    results[f"hybrid+expand[{variant}]"] = record(
        f"hybrid+expand[{variant}]",
        hybrid(embedder, sparse, hstore, "ticker+year", expand=True), hstore.count(), note)

### Against ADR-007

Read the **ceiling** column, not R@5. On 44 questions shallow recall moves in
steps of 0.023 - one question - which is far too coarse to judge a change by.
Recall at depth 200 is whether the right evidence is being surfaced at all.

In [ ]:
ledger = ev.load_runs()
prior = {r.config.retriever: r for r in ledger
         if r.config.retriever in ("dense+expand", "hybrid+expand")
         and r.config.model == MODEL and r.config.filters == "ticker+year"}

wanted = ["dense+expand", "dense+expand[fix]", "dense+expand[ctx]",
          "hybrid+expand", "hybrid+expand[fix]", "hybrid+expand[ctx]"]
picked = {**prior, **results}
table = pd.DataFrame([picked[k].row() for k in wanted if k in picked])
print(table.to_string(index=False))

print()
curves = pd.DataFrame({k: picked[k].depth_curve for k in wanted if k in picked}).T
print(curves.rename_axis("retriever").to_string())

## 7. Where it still fails

In [ ]:
embedder, store = open_store(settings, MODEL, "ctx")
_, sparse, hstore = open_hybrid(settings, MODEL, "ctx")
res = ev.evaluate(questions, hybrid(embedder, sparse, hstore, "ticker+year", expand=True),
                  limit=max(ev.DEPTHS))
df = pd.DataFrame([r.model_dump() for r in res])
print(df.groupby("question_type")["rank"].agg(n="size", found="count").to_string())
print()
print(df.groupby("ticker")["rank"].agg(n="size", found="count").to_string())

## 8. The ledger

In [ ]:
print(ev.write_leaderboard(ev.load_runs()))